In [1]:
# Notebook 06: Pillar 1 analysis + paper-ready figures and tables.
# Inputs : results/{rf,xgb,mlp}_metrics.json on Drive
# Outputs: figures/*.{png,pdf} and results/pillar1_summary.{json,tex}
# Runtime: ~30 seconds

import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/composite_resilience_framework')
RESULTS = PROJECT / 'results'
FIGURES = PROJECT / 'figures'
FIGURES.mkdir(exist_ok=True)

CATS = ['Benign', 'BruteForce', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web']
DETECTORS = [
    ('RF',  'rf_metrics.json',  'Random Forest'),
    ('XGB', 'xgb_metrics.json', 'XGBoost'),
    ('MLP', 'mlp_metrics.json', 'MLP'),
]

# Publication-grade matplotlib defaults
plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# =========================================================================
# Load all metrics
# =========================================================================
print("=" * 70)
print("LOADING METRICS")
print("=" * 70)
all_metrics = {}
for short, fname, _ in DETECTORS:
    m = json.load(open(RESULTS / fname))
    all_metrics[short] = m
    print(f"  {short}: {len(m)} seeds loaded from {fname}")

# =========================================================================
# Build aggregate statistics across seeds
# =========================================================================
def agg_metric(det_short, split, metric_key):
    runs = all_metrics[det_short]
    vals = [runs[s][split][metric_key] for s in sorted(runs.keys())]
    return np.mean(vals), np.std(vals), vals

def agg_per_class(det_short, split, cat, metric_key):
    runs = all_metrics[det_short]
    vals = [runs[s][split]['per_class'][cat][metric_key] for s in sorted(runs.keys())]
    return np.mean(vals), np.std(vals)

# =========================================================================
# TABLE 2: Aggregate detection metrics, mean ± std across seeds
# =========================================================================
print("\n" + "=" * 70)
print("TABLE 2: Aggregate test-set metrics (mean ± std, 5 seeds)")
print("=" * 70)
rows = []
for short, _, longname in DETECTORS:
    acc_m, acc_s, _ = agg_metric(short, 'test', 'accuracy')
    pm_m, pm_s, _   = agg_metric(short, 'test', 'precision_macro')
    rm_m, rm_s, _   = agg_metric(short, 'test', 'recall_macro')
    fm_m, fm_s, _   = agg_metric(short, 'test', 'f1_macro')
    pw_m, pw_s, _   = agg_metric(short, 'test', 'precision_weighted')
    rw_m, rw_s, _   = agg_metric(short, 'test', 'recall_weighted')
    fw_m, fw_s, _   = agg_metric(short, 'test', 'f1_weighted')
    rows.append({
        'Detector': longname,
        'Accuracy':         f'{acc_m:.4f} ± {acc_s:.4f}',
        'Precision (macro)':f'{pm_m:.4f} ± {pm_s:.4f}',
        'Recall (macro)':   f'{rm_m:.4f} ± {rm_s:.4f}',
        'F1 (macro)':       f'{fm_m:.4f} ± {fm_s:.4f}',
        'Precision (wtd)':  f'{pw_m:.4f} ± {pw_s:.4f}',
        'Recall (wtd)':     f'{rw_m:.4f} ± {rw_s:.4f}',
        'F1 (wtd)':         f'{fw_m:.4f} ± {fw_s:.4f}',
    })
table2 = pd.DataFrame(rows).set_index('Detector')
print("\n" + table2.to_string())

# LaTeX export for Table 2
def fmt_with_pm(mean, std, decimals=4):
    return f'{mean:.{decimals}f} $\\pm$ {std:.{decimals}f}'

latex_lines = [
    r'\begin{table}[t]',
    r'\centering',
    r'\caption{Detection metrics on the held-out test set (7{,}016{,}505 flows). '
    r'Values are mean $\pm$ standard deviation across 5 random seeds. '
    r'Macro-averaged metrics weight all 8 categories equally; weighted-averaged metrics '
    r'are weighted by class support. Tight standard deviations confirm seed-stable results.}',
    r'\label{tab:detection_metrics}',
    r'\small',
    r'\begin{tabular}{lccccc}',
    r'\toprule',
    r'Detector & Accuracy & Precision (macro) & Recall (macro) & F1 (macro) & F1 (weighted) \\',
    r'\midrule',
]
for short, _, longname in DETECTORS:
    acc_m, acc_s, _ = agg_metric(short, 'test', 'accuracy')
    pm_m, pm_s, _   = agg_metric(short, 'test', 'precision_macro')
    rm_m, rm_s, _   = agg_metric(short, 'test', 'recall_macro')
    fm_m, fm_s, _   = agg_metric(short, 'test', 'f1_macro')
    fw_m, fw_s, _   = agg_metric(short, 'test', 'f1_weighted')
    latex_lines.append(
        f'{longname} & {fmt_with_pm(acc_m,acc_s)} & {fmt_with_pm(pm_m,pm_s)} & '
        f'{fmt_with_pm(rm_m,rm_s)} & {fmt_with_pm(fm_m,fm_s)} & {fmt_with_pm(fw_m,fw_s)} \\\\'
    )
latex_lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
table2_tex = '\n'.join(latex_lines)
(RESULTS / 'table2_detection_metrics.tex').write_text(table2_tex)
print(f"\n  Saved: {RESULTS / 'table2_detection_metrics.tex'}")

# =========================================================================
# TABLE 3: Per-class F1 across detectors
# =========================================================================
print("\n" + "=" * 70)
print("TABLE 3: Per-class F1 (mean across 5 seeds)")
print("=" * 70)
perclass_data = {}
perclass_std = {}
for short, _, longname in DETECTORS:
    means, stds = {}, {}
    for cat in CATS:
        m, s = agg_per_class(short, 'test', cat, 'f1')
        means[cat] = m; stds[cat] = s
    perclass_data[longname] = means
    perclass_std[longname] = stds
table3 = pd.DataFrame(perclass_data, index=CATS)
table3['Best'] = table3.idxmax(axis=1)
print("\n" + table3.to_string(float_format='%.4f'))

# Per-class support (same in all seeds, use seed_42)
support = {cat: all_metrics['RF']['seed_42']['test']['per_class'][cat]['support']
           for cat in CATS}

# LaTeX export for Table 3
latex_lines = [
    r'\begin{table}[t]',
    r'\centering',
    r'\caption{Per-class F1 score on the test set, mean across 5 random seeds. '
    r'Support is the number of test flows per class. Highest F1 per class is shown in bold. '
    r'All detectors achieve near-perfect F1 on Mirai but degrade on minority classes '
    r'(Web, BruteForce) and exhibit a structural confusion between DDoS and DoS (Section X).}',
    r'\label{tab:per_class_f1}',
    r'\small',
    r'\begin{tabular}{lrccc}',
    r'\toprule',
    r'Class & Support & Random Forest & XGBoost & MLP \\',
    r'\midrule',
]
for cat in CATS:
    rf = perclass_data['Random Forest'][cat]
    xg = perclass_data['XGBoost'][cat]
    ml = perclass_data['MLP'][cat]
    best_val = max(rf, xg, ml)
    def fmt(v): return f'\\textbf{{{v:.4f}}}' if abs(v - best_val) < 1e-9 else f'{v:.4f}'
    latex_lines.append(f'{cat} & {support[cat]:,} & {fmt(rf)} & {fmt(xg)} & {fmt(ml)} \\\\')
latex_lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
(RESULTS / 'table3_per_class_f1.tex').write_text('\n'.join(latex_lines))
print(f"\n  Saved: {RESULTS / 'table3_per_class_f1.tex'}")

# =========================================================================
# FIGURE 4: Aggregate metrics bar chart (grouped)
# =========================================================================
print("\n" + "=" * 70)
print("FIGURE 4: Aggregate test metrics")
print("=" * 70)
fig, ax = plt.subplots(figsize=(7.2, 4.0))
metric_names = ['Accuracy', 'Macro-Precision', 'Macro-Recall', 'Macro-F1', 'Weighted-F1']
metric_keys  = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'f1_weighted']
x = np.arange(len(metric_names))
width = 0.25
colors = ['#3F6FB0', '#C0563B', '#3F8C5A']
for i, (short, _, longname) in enumerate(DETECTORS):
    means = [agg_metric(short, 'test', k)[0] for k in metric_keys]
    stds  = [agg_metric(short, 'test', k)[1] for k in metric_keys]
    ax.bar(x + (i-1)*width, means, width, yerr=stds, capsize=3,
           label=longname, color=colors[i], edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(metric_names, rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.0)
ax.set_yticks(np.arange(0, 1.05, 0.1))
ax.axhline(0.5, color='gray', linewidth=0.5, linestyle=':')
ax.legend(loc='lower right', framealpha=0.95)
ax.set_title('Detection metrics on full test set (mean ± std, 5 seeds)')
plt.tight_layout()
plt.savefig(FIGURES / 'fig4_detection_metrics.png', dpi=300)
plt.savefig(FIGURES / 'fig4_detection_metrics.pdf')
plt.show()
print(f"  Saved: figures/fig4_detection_metrics.{{png,pdf}}")

# =========================================================================
# FIGURE 4b: Per-class F1 heatmap (the interesting one)
# =========================================================================
print("\n" + "=" * 70)
print("FIGURE 4b: Per-class F1 heatmap across detectors")
print("=" * 70)
heat_data = np.array([[perclass_data[d][cat] for cat in CATS]
                       for _, _, d in DETECTORS])
det_short_names = [d for _, _, d in DETECTORS]

fig, ax = plt.subplots(figsize=(8.0, 2.6))
im = ax.imshow(heat_data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(np.arange(len(CATS))); ax.set_xticklabels(CATS, rotation=30, ha='right')
ax.set_yticks(np.arange(len(det_short_names))); ax.set_yticklabels(det_short_names)
for i in range(len(det_short_names)):
    for j in range(len(CATS)):
        v = heat_data[i, j]
        # Choose text color for readability against cell
        tcol = 'white' if v < 0.35 or v > 0.85 else 'black'
        ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=9, color=tcol)
ax.set_title('Per-class F1 across detectors (test set, mean of 5 seeds)')
cbar = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('F1')
plt.tight_layout()
plt.savefig(FIGURES / 'fig4b_per_class_heatmap.png', dpi=300)
plt.savefig(FIGURES / 'fig4b_per_class_heatmap.pdf')
plt.show()
print(f"  Saved: figures/fig4b_per_class_heatmap.{{png,pdf}}")

# =========================================================================
# FIGURE 5: Side-by-side normalized confusion matrices
# =========================================================================
print("\n" + "=" * 70)
print("FIGURE 5: Confusion matrices (row-normalized, seed 42)")
print("=" * 70)
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))
for ax_i, (short, _, longname) in enumerate(DETECTORS):
    cm = np.array(all_metrics[short]['seed_42']['test']['confusion_matrix'])
    cm_n = cm / cm.sum(axis=1, keepdims=True)
    ax = axes[ax_i]
    im = ax.imshow(cm_n, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(np.arange(len(CATS))); ax.set_xticklabels(CATS, rotation=45, ha='right')
    ax.set_yticks(np.arange(len(CATS))); ax.set_yticklabels(CATS)
    ax.set_xlabel('Predicted')
    if ax_i == 0:
        ax.set_ylabel('True')
    ax.set_title(longname)
    for i in range(len(CATS)):
        for j in range(len(CATS)):
            v = cm_n[i, j]
            if v >= 0.01:
                tcol = 'white' if v > 0.5 else 'black'
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        fontsize=7, color=tcol)
fig.suptitle('Row-normalized confusion matrices (test set, seed 42). '
             'Each row sums to 1.0; diagonal = recall.', y=1.02, fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / 'fig5_confusion_matrices.png', dpi=300)
plt.savefig(FIGURES / 'fig5_confusion_matrices.pdf')
plt.show()
print(f"  Saved: figures/fig5_confusion_matrices.{{png,pdf}}")

# =========================================================================
# DDoS↔DoS confusion summary table
# =========================================================================
print("\n" + "=" * 70)
print("DDoS↔DoS confusion table (across detectors, seed 42)")
print("=" * 70)
ddos_i = CATS.index('DDoS'); dos_i = CATS.index('DoS')
conf_rows = []
for short, _, longname in DETECTORS:
    cm = np.array(all_metrics[short]['seed_42']['test']['confusion_matrix'])
    cm_n = cm / cm.sum(axis=1, keepdims=True)
    conf_rows.append({
        'Detector': longname,
        'DDoS→DoS': f'{cm_n[ddos_i, dos_i]*100:.1f}%',
        'DoS→DDoS': f'{cm_n[dos_i, ddos_i]*100:.1f}%',
    })
conf_table = pd.DataFrame(conf_rows).set_index('Detector')
print("\n" + conf_table.to_string())

# =========================================================================
# Final summary JSON for downstream use
# =========================================================================
summary = {
    'pillar': 'Pillar 1: Detection baseline',
    'dataset': 'CICIoT2023',
    'total_test_flows': sum(support.values()),
    'seeds': 5,
    'detectors': {},
}
for short, _, longname in DETECTORS:
    summary['detectors'][longname] = {
        'macro_f1_mean':    float(agg_metric(short, 'test', 'f1_macro')[0]),
        'macro_f1_std':     float(agg_metric(short, 'test', 'f1_macro')[1]),
        'weighted_f1_mean': float(agg_metric(short, 'test', 'f1_weighted')[0]),
        'weighted_f1_std':  float(agg_metric(short, 'test', 'f1_weighted')[1]),
        'accuracy_mean':    float(agg_metric(short, 'test', 'accuracy')[0]),
        'accuracy_std':     float(agg_metric(short, 'test', 'accuracy')[1]),
        'per_class_f1_mean': {
            cat: float(perclass_data[longname][cat]) for cat in CATS
        },
    }
(RESULTS / 'pillar1_summary.json').write_text(json.dumps(summary, indent=2))
print(f"\n  Saved: results/pillar1_summary.json")

print("\n" + "=" * 70)
print("PILLAR 1 ANALYSIS COMPLETE")
print("=" * 70)
print(f"\nArtifacts on Drive:")
print(f"  Tables (LaTeX): results/table2_detection_metrics.tex")
print(f"                  results/table3_per_class_f1.tex")
print(f"  Figures:        figures/fig4_detection_metrics.{{png,pdf}}")
print(f"                  figures/fig4b_per_class_heatmap.{{png,pdf}}")
print(f"                  figures/fig5_confusion_matrices.{{png,pdf}}")
print(f"  Summary JSON:   results/pillar1_summary.json")

Mounted at /content/drive
LOADING METRICS
  RF: 5 seeds loaded from rf_metrics.json
  XGB: 5 seeds loaded from xgb_metrics.json
  MLP: 5 seeds loaded from mlp_metrics.json

TABLE 2: Aggregate test-set metrics (mean ± std, 5 seeds)

                      Accuracy Precision (macro)   Recall (macro)       F1 (macro)  Precision (wtd)     Recall (wtd)         F1 (wtd)
Detector                                                                                                                             
Random Forest  0.8572 ± 0.0001   0.8635 ± 0.0009  0.6114 ± 0.0005  0.6454 ± 0.0005  0.8454 ± 0.0002  0.8572 ± 0.0001  0.8361 ± 0.0005
XGBoost        0.7566 ± 0.0004   0.6450 ± 0.0001  0.7708 ± 0.0006  0.6473 ± 0.0002  0.8731 ± 0.0001  0.7566 ± 0.0004  0.7828 ± 0.0004
MLP            0.7331 ± 0.0022   0.6181 ± 0.0007  0.7303 ± 0.0004  0.5984 ± 0.0013  0.8704 ± 0.0005  0.7331 ± 0.0022  0.7630 ± 0.0019

  Saved: /content/drive/MyDrive/composite_resilience_framework/results/table2_detection_metrics.t

<Figure size 1080x600 with 1 Axes>

  Saved: figures/fig4_detection_metrics.{png,pdf}

FIGURE 4b: Per-class F1 heatmap across detectors


<Figure size 1200x390 with 2 Axes>

  Saved: figures/fig4b_per_class_heatmap.{png,pdf}

FIGURE 5: Confusion matrices (row-normalized, seed 42)


<Figure size 2025x660 with 3 Axes>

  Saved: figures/fig5_confusion_matrices.{png,pdf}

DDoS↔DoS confusion table (across detectors, seed 42)

              DDoS→DoS DoS→DDoS
Detector                       
Random Forest     2.7%    67.1%
XGBoost          29.9%     8.9%
MLP              33.0%     7.2%

  Saved: results/pillar1_summary.json

PILLAR 1 ANALYSIS COMPLETE

Artifacts on Drive:
  Tables (LaTeX): results/table2_detection_metrics.tex
                  results/table3_per_class_f1.tex
  Figures:        figures/fig4_detection_metrics.{png,pdf}
                  figures/fig4b_per_class_heatmap.{png,pdf}
                  figures/fig5_confusion_matrices.{png,pdf}
  Summary JSON:   results/pillar1_summary.json
